# Unify EHR Data – Parse Clinical Notes, Load Tables, and Add Keys

This notebook is the **unify_ehr_data** workflow. It:

1. Reads all 100 PDF clinical notes from the volume subfolder `clinical_notes`.
2. Parses each PDF with **ai_parse_document** (Agent Brick–style document parsing).
3. Writes the parsed results into **melissap.melissa_pang.parsed_clinical_notes**.
4. Loads **ehr.csv** and **lung_cancer_images_metadata.json** from the volume into Delta tables **ehr** and **lung_cancer_images**.
5. Adds primary key and foreign key constraints so **ehr**, **lung_cancer_images**, and **parsed_clinical_notes** can be joined on `patient_id`.

**Volume:** `/Volumes/melissap/melissa_pang/project_volume/` (root: CSV, JSON; subfolder `clinical_notes/`: PDFs).

In [59]:
# Get Spark session (injected on Databricks; for local run use profile 'tko' with serverless_compute_id=auto or cluster_id in ~/.databrickscfg)
def _get_spark(force_new=False):
    from databricks.connect import DatabricksSession
    b = DatabricksSession.builder.profile("tko").serverless()
    return b.create() if force_new else b.getOrCreate()
try:
    spark.sql("SELECT 1").collect()
except (NameError, Exception) as e:
    if isinstance(e, NameError) or "NO_ACTIVE_SESSION" in str(e) or "No active Spark session" in str(e):
        spark = _get_spark(force_new=not isinstance(e, NameError))
    else:
        raise
print("Spark session ready.")

Spark session ready.


In [60]:
# Configuration
CATALOG = "melissap"
SCHEMA = "melissa_pang"
VOLUME_BASE = "/Volumes/melissap/melissa_pang/project_volume"
VOLUME_PATH = f"{VOLUME_BASE}/clinical_notes"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.parsed_clinical_notes"

# For loading CSV/JSON and PK/FK (used later in this notebook)
EHR_CSV_PATH = f"{VOLUME_BASE}/ehr.csv"
IMAGES_JSON_PATH = f"{VOLUME_BASE}/lung_cancer_images_metadata.json"
EHR_TABLE = f"{CATALOG}.{SCHEMA}.ehr"
IMAGES_TABLE = f"{CATALOG}.{SCHEMA}.lung_cancer_images"
NOTES_TABLE = TABLE_NAME  # parsed_clinical_notes

In [61]:
# Ensure we have an active Spark session (recreate if connection was closed, e.g. after fork)
def _ensure_spark():
    from databricks.connect import DatabricksSession
    return DatabricksSession.builder.profile("tko").serverless().create()
try:
    spark.sql("SELECT 1").collect()
except Exception as e:
    if "NO_ACTIVE_SESSION" in str(e) or "No active Spark session" in str(e) or isinstance(e, NameError):
        spark = _ensure_spark()
    else:
        raise

# Ensure catalog and schema exist and set context
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Table will be created as: {TABLE_NAME}")

Table will be created as: melissap.melissa_pang.parsed_clinical_notes


In [62]:
# Read all PDFs from the volume as binary, parse with ai_parse_document, extract patient_id from path
# Uses Agent Brick–style document parsing (ai_parse_document) for the 100 clinical notes.
from pyspark.sql.functions import col, current_timestamp, expr, regexp_extract

df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(VOLUME_PATH)
)

parsed_df = (
    df
    .withColumn("parsed", expr("ai_parse_document(content)"))
    .withColumn("patient_id", regexp_extract(col("path"), r"patient_(PAT_\d+)", 1))
    .withColumn("parsed_at", current_timestamp())
    .select("path", "patient_id", "parsed", "parsed_at")
)

parsed_df.printSchema()
print(f"Parsed {parsed_df.count()} documents")

root
 |-- path: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- parsed: variant (nullable = true)
 |-- parsed_at: timestamp (nullable = false)

Parsed 100 documents


In [63]:
# Flatten parsed content to text for easier querying; keep parsed variant for full structure
from pyspark.sql.functions import expr

text_df = (
    parsed_df
    .withColumn(
        "parsed_text",
        expr("""
            concat_ws(
                '\\n\\n',
                transform(
                    try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                    element -> try_cast(element:content AS STRING)
                )
            )
        """)
    )
    .select("path", "patient_id", "parsed", "parsed_text", "parsed_at")
)

text_df.show(2, truncate=50)

+--------------------------------------------------+----------+--------------------------------------------------+--------------------------------------------------+--------------------------+
|                                              path|patient_id|                                            parsed|                                       parsed_text|                 parsed_at|
+--------------------------------------------------+----------+--------------------------------------------------+--------------------------------------------------+--------------------------+
|dbfs:/Volumes/melissap/melissa_pang/project_vol...|   PAT_006|{"document":{"elements":[{"bbox":[{"coord":[50,...|Confidential - Synthetic Clinical Note\n\nCLINI...|2026-03-05 22:21:09.913039|
|dbfs:/Volumes/melissap/melissa_pang/project_vol...|   PAT_002|{"document":{"elements":[{"bbox":[{"coord":[50,...|Confidential - Synthetic Clinical Note\n\nCLINI...|2026-03-05 22:21:09.913039|
+----------------------------------

In [64]:
# Parse structured fields from parsed_text (matches format from synthetic data notebook)
from pyspark.sql.functions import col, regexp_extract

structured_df = (
    text_df
    .withColumn("note_patient_id", regexp_extract(col("parsed_text"), r"Patient ID:\s*(\S+)", 1))
    .withColumn("note_date", regexp_extract(col("parsed_text"), r"Date:\s*([^\n]+)", 1))
    .withColumn("encounter", regexp_extract(col("parsed_text"), r"Encounter:\s*(\d+)", 1))
    .withColumn("chief_complaint", regexp_extract(col("parsed_text"), r"Chief Complaint:\s*([^\n]+)", 1))
    .withColumn("assessment", regexp_extract(col("parsed_text"), r"Assessment:\s*([^\n]+)", 1))
    .withColumn("plan", regexp_extract(col("parsed_text"), r"Plan:\s*([^\n]+)", 1))
    .withColumn("next_visit", regexp_extract(col("parsed_text"), r"Next visit:\s*([^\n]+)", 1))
)

structured_df.select(
    "patient_id", "note_patient_id", "note_date", "encounter",
    "chief_complaint", "assessment", "plan", "next_visit"
).show(3, truncate=30)

+----------+---------------+----------+---------+-----------------------------+------------------------------+------------------------------+----------+
|patient_id|note_patient_id| note_date|encounter|              chief_complaint|                    assessment|                          plan|next_visit|
+----------+---------------+----------+---------+-----------------------------+------------------------------+------------------------------+----------+
|   PAT_006|        PAT_006|2024-12-24|       24|   Routine oncology follow-up|New nodule noted; recommend...|Organization people informa...|2026-03-05|
|   PAT_002|        PAT_002|2025-01-27|       68|      Fatigue and weight loss|No interval change. Surveil...|Wrong figure perform partic...|2026-03-05|
|   PAT_017|        PAT_017|2025-07-27|       78|Cough and shortness of breath|Stable lung malignancy. Con...|Participant dream citizen n...|2026-03-05|
+----------+---------------+----------+---------+-----------------------------+---

In [65]:
# Write to Unity Catalog table (overwrite for full refresh)
# Table includes path, patient_id, parsed, parsed_text, parsed_at, plus note_patient_id, note_date, encounter, chief_complaint, assessment, plan, next_visit
structured_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(TABLE_NAME)
print(f"Table saved: {TABLE_NAME}")
spark.table(TABLE_NAME).count()

Table saved: melissap.melissa_pang.parsed_clinical_notes


100

### Create the **unify_ehr_data** job in Databricks

To run this notebook as a job, mirror this project in the Databricks workspace using **Repos** (Git). Then the job can reference the notebook in the repo.

**See [DATABRICKS_REPO_SETUP.md](DATABRICKS_REPO_SETUP.md)** in the project root for:

1. Pushing this project to Git and creating a Databricks Repo that clones it  
2. Getting the notebook path in the workspace  
3. Setting `NOTEBOOK_IN_WORKSPACE_PATH` in the cell below and re-running it to create/update the job  

After you pull in Databricks, edits you make in Cursor and push to Git will be reflected when you pull in the repo.

In [66]:
# Create or update the workflow job 'unify_ehr_data' (run after mirroring via Repos — see DATABRICKS_REPO_SETUP.md)
# Set to your notebook path in the workspace (no .ipynb). Example after Repo is created:
#   "/Workspace/Repos/your.email@domain/claude-playground/3.unify_ehr_data"
NOTEBOOK_IN_WORKSPACE_PATH = None

if NOTEBOOK_IN_WORKSPACE_PATH is None:
    print(
        "Job creation skipped. To create the job:\n"
        "  1. Mirror this project in Databricks using Repos — see DATABRICKS_REPO_SETUP.md.\n"
        "  2. Set NOTEBOOK_IN_WORKSPACE_PATH above to the notebook path (Copy path in Databricks).\n"
        "  3. Re-run this cell."
    )
else:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.jobs import Task, NotebookTask, Source

    w = WorkspaceClient(profile="DEFAULT")
    existing = list(w.jobs.list(name="unify_ehr_data", limit=5))
    tasks = [
        Task(
            task_key="parse_clinical_notes",
            description="Parse 100 PDF clinical notes and load to parsed_clinical_notes table",
            notebook_task=NotebookTask(
                notebook_path=NOTEBOOK_IN_WORKSPACE_PATH,
                source=Source.WORKSPACE,
            ),
        )
    ]
    if existing:
        job_id = existing[0].job_id
        w.jobs.reset(job_id=job_id, new_settings={"name": "unify_ehr_data", "tasks": [t.as_dict() for t in tasks]})
        print(f"Updated workflow job 'unify_ehr_data' (job_id={job_id})")
    else:
        job = w.jobs.create(name="unify_ehr_data", tasks=tasks)
        print(f"Created workflow job 'unify_ehr_data' (job_id={job.job_id})")

Job creation skipped. To create the job:
  1. Mirror this project in Databricks using Repos — see DATABRICKS_REPO_SETUP.md.
  2. Set NOTEBOOK_IN_WORKSPACE_PATH above to the notebook path (Copy path in Databricks).
  3. Re-run this cell.


## Load EHR CSV and images JSON to Delta; add primary/foreign keys

Load **ehr** and **lung_cancer_images** from the volume, then add PRIMARY KEY and FOREIGN KEY so all three tables join on `patient_id`.

### 1. Load EHR CSV (primary key: patient_id)

In [67]:
# Read CSV from volume and write to Delta; patient_id is unique per row (one row per patient)
ehr_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(EHR_CSV_PATH)
)
ehr_df.printSchema()
ehr_df.write.format("delta").mode("overwrite").saveAsTable(EHR_TABLE)
print(f"Loaded {ehr_df.count()} rows into {EHR_TABLE}")

root
 |-- patient_id: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- sex: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- medications: string (nullable = true)
 |-- allergies: string (nullable = true)
 |-- last_visit: date (nullable = true)

Loaded 35 rows into melissap.melissa_pang.ehr


In [68]:
# Add primary key on patient_id (parent table for joins)
from pyspark.sql.utils import AnalysisException
try:
    spark.sql(f"ALTER TABLE {EHR_TABLE} ADD CONSTRAINT ehr_pk PRIMARY KEY (patient_id)")
    print("Added PRIMARY KEY (patient_id) on ehr")
except AnalysisException as e:
    if "already exists" in str(e).lower() or "constraint" in str(e).lower():
        print("ehr_pk already exists; skipping.")
    else:
        raise

ehr_pk already exists; skipping.


### 2. Load lung cancer images JSON (primary key: image_id, foreign key: patient_id → ehr)

In [69]:
# JSON has root { "patients": [...], "images": [ { image_id, patient_id, modality, ... } ] }
# Read as binary, parse in Python, then create DataFrame from the "images" array.
import json

binary_df = spark.read.format("binaryFile").load(IMAGES_JSON_PATH)
content = binary_df.select("content").first()[0]
data = json.loads(content.decode("utf-8"))
images_df = spark.createDataFrame(data["images"])
images_df.printSchema()
images_df.write.format("delta").mode("overwrite").saveAsTable(IMAGES_TABLE)
print(f"Loaded {images_df.count()} rows into {IMAGES_TABLE}")

root
 |-- body_site: string (nullable = true)
 |-- finding: string (nullable = true)
 |-- image_id: string (nullable = true)
 |-- modality: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- slice_count: long (nullable = true)
 |-- study_date: string (nullable = true)

Loaded 151 rows into melissap.melissa_pang.lung_cancer_images


In [70]:
# Add primary key and foreign key to ehr
try:
    spark.sql(f"ALTER TABLE {IMAGES_TABLE} ADD CONSTRAINT images_pk PRIMARY KEY (image_id)")
    spark.sql(f"ALTER TABLE {IMAGES_TABLE} ADD CONSTRAINT images_fk_patient "
              f"FOREIGN KEY (patient_id) REFERENCES {EHR_TABLE}(patient_id)")
    print("Added PRIMARY KEY (image_id) and FOREIGN KEY (patient_id) -> ehr(patient_id)")
except AnalysisException as e:
    if "already exists" in str(e).lower() or "constraint" in str(e).lower():
        print("images_pk / images_fk_patient already exist; skipping.")
    else:
        raise

images_pk / images_fk_patient already exist; skipping.
